# B.O.S.S. — RKNN PC-Simulator: YOLO11n / YOLO11s su RK3588

Simulazione inferenza NPU RK3588 via `rknn-toolkit2` in **PC-simulator mode** (nessuna board fisica).
Copre i task roadmap **s4-1** (installazione RKNN Toolkit2 PC-sim), **s4-2** (operator compatibility NPU-native vs CPU-fallback),
**t1-1** (operator check zero-shot YOLO11-n) e **t1-2** (stima latenza FP16 baseline), estesi anche a YOLO11-s e a INT8.

**Pipeline:**
1. Install `rknn-toolkit2` (PyPI, wheel x86_64) + `ultralytics`, con pin `torch==2.4.0` (vincolo rknn-toolkit2, vedi Cella 1)
2. Export `yolo11n.pt` / `yolo11s.pt` → ONNX 640x640 (stessi pesi COCO pretrained dei report esistenti)
3. Convert ONNX → RKNN, `target_platform='rk3588'`, build FP16 e INT8 (calibrazione da `CALIB_DIR`)
4. `rknn.eval_perf()` → stima latenza ms/frame, confronto budget TC-004 (33 ms end-to-end)
5. Operator compatibility check (NPU-native vs CPU-fallback), se esposto dalla toolkit
6. Salvataggio risultati in JSON, schema coerente con `report YOLO11n.md` / `report YOLO11s.md`

Hailo **non** è trattato qui (Dataflow Compiler richiede download manuale da Hailo Developer Zone) — notebook separato.

In [ ]:
# ============================================================
# Cella 1 — Install rknn-toolkit2 (PyPI, wheel x86_64) + ultralytics
# ============================================================
# rknn-toolkit2 2.3.2 pin: torch<=2.4.0,>=1.10.1. L'immagine Kaggle di base ha
# torch 2.10.0+cpu preinstallato: se lo si lascia risolvere in due pip separati,
# rknn-toolkit2 lo declassa a 2.4.0 e la reinstallazione successiva di ultralytics
# lo fa risalire a 2.10.0, rompendo di nuovo il vincolo di rknn-toolkit2 (visto in
# esecuzione: conflitti torch/torchvision/torchaudio in loop). Fix: fissare torch/
# torchvision/torchaudio a una tripletta compatibile con rknn-toolkit2 in un unico
# passo (build CPU, coerente con quanto già presente), poi installare il resto
# senza che tocchi più torch (i vincoli di ultralytics sono già soddisfatti).
import sys
print(f"Python: {sys.version}")

%pip install --quiet "torch==2.4.0" "torchvision==0.19.0" "torchaudio==2.4.0" --index-url https://download.pytorch.org/whl/cpu
%pip install --quiet rknn-toolkit2 ultralytics onnx onnx-tool

In [ ]:
# ============================================================
# Cella 2 — Import
# ============================================================
import os
import json
import shutil
from pathlib import Path

from ultralytics import YOLO
from rknn.api import RKNN

try:
    import boss_eval_utils as beu  # Kaggle: utility script auto-mounted in sys.path
except ModuleNotFoundError:
    sys.path.insert(0, "/kaggle/usr/lib/notebooks/lorenzoverdura/boss_eval_utils")
    import boss_eval_utils as beu
print(f"boss_eval_utils caricato da: {beu.__file__}")

In [ ]:
# ============================================================
# Cella 3 — Configurazione
# ============================================================
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

if IS_KAGGLE:
    KAGGLE_MODEL_DIR = Path("/kaggle/input/models/ultralytics/yolo11/pytorch/default/1")
else:
    KAGGLE_MODEL_DIR = Path("/home/lorenzoverdura8/BOSS_CODE")

OUTPUT_DIR = Path("/kaggle/working")
RKNN_OUTPUT = OUTPUT_DIR / "RKNN_SIM_OUTPUT"
if RKNN_OUTPUT.exists():
    shutil.rmtree(RKNN_OUTPUT)
RKNN_OUTPUT.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 640
TARGET_PLATFORM = "rk3588"
LATENCY_BUDGET_MS = 33  # TC-004, OR4_tasks.json _meta.latency_budget_ms

# Cartella di calibrazione INT8, configurabile: puntarla a un Kaggle Dataset con
# un piccolo set di immagini rappresentative (es. frame D435i o recordings BOSS).
# Se non esiste/vuota, si usa in fallback un piccolo set di immagini bundlate con
# ultralytics (solo per non bloccare l'esecuzione end-to-end del notebook).
CALIB_DIR = Path("/kaggle/input/boss-rknn-calib")


def resolve_weights(fname):
    cand = KAGGLE_MODEL_DIR / fname
    if cand.exists():
        return str(cand)
    if IS_KAGGLE:
        for p in Path("/kaggle/input").rglob(fname):
            return str(p)
    return fname


MODELS = [
    {"name": "YOLO11n", "weights": resolve_weights("yolo11n.pt")},
    {"name": "YOLO11s", "weights": resolve_weights("yolo11s.pt")},
]
PRECISIONS = [
    {"label": "FP16", "do_quantization": False},
    {"label": "INT8", "do_quantization": True},
]

print(f"Ambiente: {'Kaggle' if IS_KAGGLE else 'Locale'}")
for m in MODELS:
    print(f"Modello {m['name']:8s}: {m['weights']} — esiste: {Path(m['weights']).exists()}")
print(f"IMG_SIZE={IMG_SIZE} | TARGET_PLATFORM={TARGET_PLATFORM} | LATENCY_BUDGET_MS={LATENCY_BUDGET_MS}")
print(f"CALIB_DIR={CALIB_DIR} — esiste: {CALIB_DIR.exists()}")

In [ ]:
# ============================================================
# Cella 4 — Set di calibrazione INT8 (lista immagini + dataset.txt per RKNN)
# ============================================================
def prepare_calib_dataset(calib_dir, out_txt_path, n_max=100):
    """Cerca immagini sotto calib_dir; fallback su asset bundlati ultralytics se assente/vuota."""
    imgs = []
    if calib_dir.exists():
        imgs = sorted(
            p for p in calib_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png")
        )[:n_max]

    used_fallback = False
    if not imgs:
        used_fallback = True
        from ultralytics.utils import ASSETS
        imgs = sorted(
            p for p in Path(ASSETS).glob("*")
            if p.suffix.lower() in (".jpg", ".jpeg", ".png")
        )
        print(f"[calib] CALIB_DIR vuota/assente: fallback su {len(imgs)} immagini bundlate ultralytics "
              "(NON rappresentative del dominio BOSS — solo per non bloccare l'esecuzione).")

    if not imgs:
        raise FileNotFoundError("Nessuna immagine di calibrazione disponibile (né CALIB_DIR né fallback).")

    out_txt_path.write_text("\n".join(str(p) for p in imgs) + "\n", encoding="utf-8")
    return imgs, used_fallback


CALIB_LIST_TXT = RKNN_OUTPUT / "calib_dataset.txt"
calib_imgs, calib_is_fallback = prepare_calib_dataset(CALIB_DIR, CALIB_LIST_TXT)
print(f"Immagini calibrazione: {len(calib_imgs)} | fallback={calib_is_fallback} | lista: {CALIB_LIST_TXT}")

In [ ]:
# ============================================================
# Cella 5 — Export ONNX 640x640 + GMACs/parametri (stesso metodo dei report esistenti)
# ============================================================
def export_onnx(weights_path, model_name, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    export_pt = out_dir / Path(str(weights_path)).name
    shutil.copy(weights_path, export_pt)
    model = YOLO(str(export_pt))
    onnx_path = Path(model.export(format="onnx", imgsz=IMG_SIZE, opset=13,
                                   dynamic=False, simplify=False))
    eff = beu.count_onnx_flops_params(onnx_path)
    print(f"{model_name}: ONNX → {onnx_path.name} | GMACs={eff['gmacs']:.3f} | "
          f"params={eff['params']/1e6:.3f} M")
    return onnx_path, eff

In [ ]:
# ============================================================
# Cella 6 — Operator compatibility check (NPU-native vs CPU-fallback), se esposto dalla toolkit
# ============================================================
def check_operator_compatibility(rknn, perf_result):
    """Deriva NPU-native vs CPU-fallback dal dettaglio per-operatore di eval_perf(),
    se la versione di rknn-toolkit2 installata lo espone. Ritorna None altrimenti
    (task s4-2: 'se disponibile nella toolkit')."""
    detail = None
    if isinstance(perf_result, dict):
        for key in ("detail", "layers", "perf_detail", "cost_detail"):
            if perf_result.get(key):
                detail = perf_result[key]
                break
    if not detail:
        return None

    def _target(d):
        return str(d.get("target", d.get("Target", ""))).upper()

    def _op_name(d):
        return d.get("op_name", d.get("OpName", d.get("type", "?")))

    total = len(detail)
    npu_ops = [d for d in detail if _target(d) == "NPU"]
    cpu_ops = [d for d in detail if _target(d) == "CPU"]
    if total == 0:
        return None
    return {
        "total_ops": total,
        "npu_native_ops": len(npu_ops),
        "cpu_fallback_ops": len(cpu_ops),
        "npu_native_pct": round(100 * len(npu_ops) / total, 2),
        "cpu_fallback_pct": round(100 * len(cpu_ops) / total, 2),
        "cpu_fallback_op_names": sorted(set(_op_name(d) for d in cpu_ops)),
    }

In [ ]:
# ============================================================
# Cella 7 — Convert ONNX → RKNN, build FP16/INT8, eval_perf() in PC-simulator mode
# ============================================================
def build_and_eval_rknn(onnx_path, model_name, precision_label, do_quantization, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    rknn = RKNN(verbose=False)

    ret = rknn.config(mean_values=[[0, 0, 0]], std_values=[[255, 255, 255]],
                       target_platform=TARGET_PLATFORM)
    if ret != 0:
        raise RuntimeError(f"{model_name}/{precision_label}: rknn.config fallito ({ret})")

    ret = rknn.load_onnx(model=str(onnx_path))
    if ret != 0:
        raise RuntimeError(f"{model_name}/{precision_label}: rknn.load_onnx fallito ({ret})")

    build_kwargs = {"do_quantization": do_quantization}
    if do_quantization:
        build_kwargs["dataset"] = str(CALIB_LIST_TXT)
    ret = rknn.build(**build_kwargs)
    if ret != 0:
        raise RuntimeError(f"{model_name}/{precision_label}: rknn.build fallito ({ret})")

    rknn_path = out_dir / f"{model_name.lower()}_{precision_label.lower()}_{TARGET_PLATFORM}.rknn"
    rknn.export_rknn(str(rknn_path))

    ret = rknn.init_runtime()  # target=None → PC-simulator, nessuna board fisica
    if ret != 0:
        raise RuntimeError(f"{model_name}/{precision_label}: init_runtime (simulator) fallito ({ret})")

    perf_result = rknn.eval_perf(is_print=True)
    latency_ms = None
    if isinstance(perf_result, dict):
        for key in ("total_time", "time", "latency", "total_ms"):
            if perf_result.get(key) is not None:
                latency_ms = float(perf_result[key])
                break

    operator_check = check_operator_compatibility(rknn, perf_result)

    rknn.release()

    return {
        "rknn_path": str(rknn_path),
        "latency_ms": latency_ms,
        "perf_result_raw": perf_result if isinstance(perf_result, dict) else None,
        "operator_check": operator_check,
    }

In [ ]:
# ============================================================
# Cella 8 — Esecuzione: export ONNX + build/eval RKNN per ogni modello x precisione
# ============================================================
results = []
for m in MODELS:
    model_root = RKNN_OUTPUT / m["name"]
    onnx_path, eff = export_onnx(m["weights"], m["name"], model_root / "onnx")

    for prec in PRECISIONS:
        print(f"\n{'='*64}\n### {m['name']} — {prec['label']}\n{'='*64}")
        rknn_out = build_and_eval_rknn(
            onnx_path, m["name"], prec["label"], prec["do_quantization"],
            model_root / "rknn" / prec["label"].lower(),
        )
        results.append({
            "modello": m["name"],
            "pesi": Path(str(m["weights"])).name,
            "gmacs_onnx": round(eff["gmacs"], 3),
            "parametri_M": round(eff["params"] / 1e6, 3),
            "dimensione_immagine": f"{IMG_SIZE}x{IMG_SIZE}",
            "target_platform": TARGET_PLATFORM,
            "precisione": prec["label"],
            "quantizzazione_int8": prec["do_quantization"],
            "calib_dir": str(CALIB_DIR) if prec["do_quantization"] else None,
            "calib_immagini": len(calib_imgs) if prec["do_quantization"] else None,
            "calib_fallback_bundlato": calib_is_fallback if prec["do_quantization"] else None,
            "latenza_stimata_ms": rknn_out["latency_ms"],
            "budget_tc004_ms": LATENCY_BUDGET_MS,
            "entro_budget_tc004": (
                rknn_out["latency_ms"] <= LATENCY_BUDGET_MS
                if rknn_out["latency_ms"] is not None else None
            ),
            "operator_check": rknn_out["operator_check"],
            "rknn_path": rknn_out["rknn_path"],
        })

print(f"\nTotale combinazioni valutate: {len(results)}")

In [ ]:
# ============================================================
# Cella 9 — Salvataggio JSON risultati (schema coerente con i report YOLO11n/YOLO11s)
# ============================================================
output_json = {
    "meta": {
        "toolkit": "rknn-toolkit2",
        "modalita": "PC-simulator (nessuna board fisica)",
        "target_platform": TARGET_PLATFORM,
        "budget_tc004_ms": LATENCY_BUDGET_MS,
        "note": "Latenza stimata dal simulatore RKNN-Toolkit2, errore atteso ±20% (task s4-1). "
                "Operator compatibility check presente solo se esposto dalla versione toolkit installata (task s4-2).",
    },
    "risultati": results,
}

json_path = RKNN_OUTPUT / "rknn_sim_results.json"
json_path.write_text(json.dumps(output_json, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Risultati salvati in: {json_path}")

for r in results:
    lat = f"{r['latenza_stimata_ms']:.2f} ms" if r["latenza_stimata_ms"] is not None else "n/d"
    entro = r["entro_budget_tc004"]
    entro_str = "n/d" if entro is None else ("OK" if entro else "SUPERATO")
    print(f"{r['modello']:8s} | {r['precisione']:4s} | GMACs={r['gmacs_onnx']:.3f} | "
          f"latenza={lat} | budget TC-004={entro_str}")